plan: Given the importance attribution of each explanation function, aggregate over trials by dividing each pixel value within a trial by the norm of the entire trial(to make each trial have distance 1 from the origin).
This makes each trial equally important while preserving the sign of each attribution value

In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson
import pickle

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def load_explanations_gradshap(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    return gradshap

In [ ]:
def load_ch_names(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    return ch_names

In [ ]:
def get_channel_importances(explanations, ch_names, take_abs=False):


    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial
    
    channel_importances = np.zeros(len(ch_names))

    for trial_idx, trial_normed in enumerate(explanations_normed):
        if take_abs:
            channel_importances += np.mean(np.abs(trial_normed), axis=1)
        else:
            channel_importances += np.mean(trial_normed, axis=1)
    
    channel_importances_dic = {ch_names[i]: channel_importances[i] for i in range(len(ch_names))}

    return channel_importances_dic

In [ ]:
def create_index_groups(n_samples, subject_index, group_size=100):
    """
    Create index groups for a given subject based on uncertainty values.
    
    Parameters:
    -----------
    uncertainties : array-like
        The uncertainties array for the subject
    subject_index : int
        Index of the subject
    group_size : int, default=100
        Size of each group
    
    Returns:
    --------
    dict
        Dictionary with subject_index as key and array of boolean index groups as value
    """
    index_groups_all = {}
    index_groups_subject = []
    
    start = 0
    while start < n_samples:
        end = min(start + group_size, n_samples-20)
        
        index_group = np.zeros(n_samples, dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    

    return index_groups_subject

In [ ]:
index_groups = create_index_groups(explanations.shape[0], 2, group_size=100)

In [ ]:
def get_channel_importances_time_indexed(explanations, ch_names, index_groups, take_abs=False):
    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial_normed = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial_normed

    #channel_importances = np.zeros(( len(index_groups),len(ch_names)))
    channel_importances = []
    for i,index_group in enumerate(index_groups):
        explanations_index_group = explanations_normed[index_group]
        channel_importances_index_group = np.zeros(len(ch_names))
        
        for trial_idx, trial_normed in enumerate(explanations_index_group):
            if take_abs:
                # taking the mean within a trial is fine as different time poins are supposed to be more important than other
                channel_importances_index_group += np.mean(np.abs(trial_normed), axis=1)
            else:
                channel_importances_index_group += np.mean(trial_normed, axis=1)
    
        channel_importances_dic = {ch_names[i]: channel_importances_index_group[i] for i in range(len(ch_names))}
        channel_importances.append(channel_importances_dic)
    

    return channel_importances
    


In [ ]:
ch_names = load_ch_names(2)
explanations_gradshap = load_explanations_gradshap(2)
channel_importances_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=False)
channel_importances_abs_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=True)

In [ ]:
def get_top_k_keys(d, k, reverse=True):

    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=reverse)[:k]
    return top_k_keys

In [ ]:
print(get_top_k_keys(channel_importances_gradshap, 10))
print(get_top_k_keys(channel_importances_gradshap, 10, reverse=False))
print(get_top_k_keys(channel_importances_abs_gradshap, 10))

In [ ]:
channel_importances_gradshap_time_indexed = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=False)
channel_importances_gradshap_time_indexed_abs = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=True)

In [ ]:
for index_group in channel_importances_time_indexed:
    print(get_top_k_keys(index_group, 10))

In [ ]:
for index_group in channel_importances_time_indexed:
    print(get_top_k_keys(index_group, 10, reverse=False))

In [ ]:
for index_group in channel_importances_time_indexed_abs:
    print(get_top_k_keys(index_group, 10))

while most important channels (5 most important) seem to be somewhat robust, this claim can not be made for the lower channels,
where importance can quickly shift

## compare results when saliency is used as explanation function

In [ ]:
ch_names = load_ch_names(2)
explanations_saliency = load_explanations_saliency(2)
channel_importances_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=False)
channel_importances_abs_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=True)

In [ ]:
print(get_top_k_keys(channel_importances_gradshap, 10))
print(get_top_k_keys(channel_importances_saliency, 10))

print(get_top_k_keys(channel_importances_gradshap, 10, reverse=False))
print(get_top_k_keys(channel_importances_saliency, 10, reverse=False))

print(get_top_k_keys(channel_importances_abs_gradshap, 10))
print(get_top_k_keys(channel_importances_abs_saliency, 10))

In [ ]:
channel_importances_saliency_time_indexed = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=False)
channel_importances_saliency_time_indexed_abs = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=True)

In [ ]:
for index_group in channel_importances_gradshap_time_indexed:
    print(get_top_k_keys(index_group, 10))

for index_group in channel_importances_saliency_time_indexed:
    print(get_top_k_keys(index_group, 10))

In [ ]:
for index_group in channel_importances_gradshap_time_indexed_abs:
    print(get_top_k_keys(index_group, 10))

for index_group in channel_importances_saliency_time_indexed_abs:
    print(get_top_k_keys(index_group, 10))

# store results for all subjects

In [ ]:
all_subject_channel_importances_gradshap = {}
all_subject_channel_importances_gradshap_abs = {}
all_subject_channel_importances_saliency = {}
all_subject_channel_importances_saliency_abs = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    ch_names = load_ch_names(subject_index)
    explanations_gradshap = load_explanations_gradshap(subject_index)
    channel_importances_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=False)
    channel_importances_abs_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=True)

    explanations_saliency = load_explanations_saliency(subject_index)
    channel_importances_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=False)
    channel_importances_abs_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=True)
    
    all_subject_channel_importances_gradshap[subject_index] = channel_importances_gradshap
    all_subject_channel_importances_gradshap_abs[subject_index] = channel_importances_abs_gradshap
    all_subject_channel_importances_saliency[subject_index] = channel_importances_saliency
    all_subject_channel_importances_saliency_abs[subject_index] = channel_importances_abs_saliency

np.save("all_subject_channel_importances_gradshap.npy", all_subject_channel_importances_gradshap)
np.save("all_subject_channel_importances_gradshap_abs.npy", all_subject_channel_importances_gradshap_abs)
np.save("all_subject_channel_importances_saliency.npy", all_subject_channel_importances_saliency)
np.save("all_subject_channel_importances_saliency_abs.npy", all_subject_channel_importances_saliency_abs)
